## Leakage checkk information

In [1]:
import json
import os
from collections import defaultdict

QO_DIR = os.path.join("..", "..", "Outputs", "qo")
DATASETS = ["PHEE", "CaseReportBench", "DiscourseEE", "MACCROBAT"]
SPLIT = "dev"

results = {}

for ds in DATASETS:
    fname = f"optimized_loqa-{ds}-{SPLIT}-gpt-oss-120b-zs-v0.json"
    fpath = os.path.join(QO_DIR, ds, fname)

    with open(fpath) as f:
        data = json.load(f)

    n_samples = len(data)
    n_leaked_iters = 0
    n_total_iters = 0
    n_samples_with_any_leak = 0
    leak_by_iter = defaultdict(lambda: {"total": 0, "leaked": 0})

    for record in data:
        sample_leaked = False
        for trace in record.get("trace_history", []):
            it = trace.get("iteration_no", -1)
            lc = trace.get("lc_info", {})
            is_leaked = lc.get("is_leaked") in ("yes", 1, True)

            n_total_iters += 1
            leak_by_iter[it]["total"] += 1

            if is_leaked:
                n_leaked_iters += 1
                leak_by_iter[it]["leaked"] += 1
                sample_leaked = True

        if sample_leaked:
            n_samples_with_any_leak += 1

    results[ds] = {
        "n_samples": n_samples,
        "n_total_iters": n_total_iters,
        "n_leaked_iters": n_leaked_iters,
        "n_samples_with_any_leak": n_samples_with_any_leak,
        "leak_by_iter": dict(leak_by_iter),
    }

    pct_iters = n_leaked_iters / n_total_iters * 100 if n_total_iters else 0
    pct_samples = n_samples_with_any_leak / n_samples * 100 if n_samples else 0
    print(f"{'=' * 50}")
    print(f"{ds} ({SPLIT})")
    print(f"  Samples:           {n_samples}")
    print(f"  Total iterations:  {n_total_iters}")
    print(f"  Leaked iterations: {n_leaked_iters} ({pct_iters:.1f}%)")
    print(f"  Samples w/ leak:   {n_samples_with_any_leak} ({pct_samples:.1f}%)")
    print(f"  By iteration:")
    for it in sorted(leak_by_iter):
        s = leak_by_iter[it]
        p = s["leaked"] / s["total"] * 100 if s["total"] else 0
        print(f"    iter {it}: {s['leaked']:4d} / {s['total']:4d} leaked ({p:.1f}%)")

PHEE (dev)
  Samples:           500
  Total iterations:  644
  Leaked iterations: 272 (42.2%)
  Samples w/ leak:   223 (44.6%)
  By iteration:
    iter 0:  212 /  500 leaked (42.4%)
    iter 1:   33 /   82 leaked (40.2%)
    iter 2:   18 /   39 leaked (46.2%)
    iter 3:    7 /   20 leaked (35.0%)
    iter 4:    2 /    3 leaked (66.7%)
CaseReportBench (dev)
  Samples:           350
  Total iterations:  505
  Leaked iterations: 280 (55.4%)
  Samples w/ leak:   219 (62.6%)
  By iteration:
    iter 0:  207 /  350 leaked (59.1%)
    iter 1:   43 /   92 leaked (46.7%)
    iter 2:   19 /   39 leaked (48.7%)
    iter 3:    8 /   17 leaked (47.1%)
    iter 4:    3 /    7 leaked (42.9%)
DiscourseEE (dev)
  Samples:           500
  Total iterations:  1104
  Leaked iterations: 516 (46.7%)
  Samples w/ leak:   323 (64.6%)
  By iteration:
    iter 0:  252 /  500 leaked (50.4%)
    iter 1:  124 /  281 leaked (44.1%)
    iter 2:   70 /  173 leaked (40.5%)
    iter 3:   51 /  114 leaked (44.7%)
    it

In [2]:
import random

random.seed(42)
N_PER_DATASET = 100

no_leak_pool = {ds: [] for ds in DATASETS}    # checker said NO leak
leaked_pool  = {ds: [] for ds in DATASETS}    # checker said YES leak (corrected)

for ds in DATASETS:
    fname = f"optimized_loqa-{ds}-{SPLIT}-gpt-oss-120b-zs-v0.json"
    fpath = os.path.join(QO_DIR, ds, fname)
    with open(fpath) as f:
        data = json.load(f)

    for record in data:
        context = record.get("context", "")
        gt_args = record.get("processed_gt_args", [])
        sid = record.get("serial-number", "")
        role = record.get("role", "")

        for trace in record.get("trace_history", []):
            lc = trace.get("lc_info", {})
            is_leaked = lc.get("is_leaked") in ("yes", 1, True)
            it = trace.get("iteration_no", -1)
            questions = trace.get("loqa_questions", [])

            entry = {
                "dataset": ds,
                "serial_number": sid,
                "role": role,
                "iteration": it,
                "context": context,
                "ground_truth": gt_args,
                "questions": questions,
            }

            if is_leaked:
                entry["previous_questions"] = lc.get("previous_questions", [])
                entry["corrected_questions"] = lc.get("final_questions", [])
                leaked_pool[ds].append(entry)
            else:
                no_leak_pool[ds].append(entry)

# --- Sample 25 per dataset ---
fn_samples = []  # false-negative check: checker said no leak
ca_samples = []  # correction-accuracy check: checker said leak & corrected

for ds in DATASETS:
    fn_samples.extend(random.sample(no_leak_pool[ds], min(N_PER_DATASET, len(no_leak_pool[ds]))))
    ca_samples.extend(random.sample(leaked_pool[ds],  min(N_PER_DATASET, len(leaked_pool[ds]))))

print(f"False-Negative annotation set: {len(fn_samples)} samples")
for ds in DATASETS:
    print(f"  {ds}: {sum(1 for s in fn_samples if s['dataset'] == ds)}")

print(f"\nCorrection-Accuracy annotation set: {len(ca_samples)} samples")
for ds in DATASETS:
    print(f"  {ds}: {sum(1 for s in ca_samples if s['dataset'] == ds)}")

# --- Save to JSON ---
OUT_DIR = os.path.join("..", "..", "Outputs", "qo", "leakage_annotation")
os.makedirs(OUT_DIR, exist_ok=True)

fn_path = os.path.join(OUT_DIR, "false_negative_check_100.json")
ca_path = os.path.join(OUT_DIR, "correction_accuracy_check_100.json")

with open(fn_path, "w") as f:
    json.dump(fn_samples, f, indent=2)
with open(ca_path, "w") as f:
    json.dump(ca_samples, f, indent=2)

print(f"\nSaved to:\n  {os.path.abspath(fn_path)}\n  {os.path.abspath(ca_path)}")

False-Negative annotation set: 400 samples
  PHEE: 100
  CaseReportBench: 100
  DiscourseEE: 100
  MACCROBAT: 100

Correction-Accuracy annotation set: 400 samples
  PHEE: 100
  CaseReportBench: 100
  DiscourseEE: 100
  MACCROBAT: 100

Saved to:
  /dartfs/rc/lab/S/SinghN/omar/LoQA/Outputs/qo/leakage_annotation/false_negative_check_100.json
  /dartfs/rc/lab/S/SinghN/omar/LoQA/Outputs/qo/leakage_annotation/correction_accuracy_check_100.json


In [3]:
print("=" * 60)
print("SAMPLE: False-Negative Check (checker said NO leak)")
print("Task: Does the question actually leak the ground truth?")
print("=" * 60)
for s in fn_samples[:2]:
    print(f"\n[{s['dataset']}] {s['serial_number']} | iter {s['iteration']} | role: {s['role']}")
    print(f"  Context:      {s['context'][:120]}...")
    print(f"  Ground Truth: {s['ground_truth']}")
    print(f"  Question(s):  {s['questions']}")
    print(f"  YOUR LABEL:   [ ] no_leak  [ ] actually_leaked")

print("\n\n" + "=" * 60)
print("SAMPLE: Correction-Accuracy Check (checker said LEAK, corrected)")
print("Task: Is the corrected question valid (no longer leaks & still makes sense)?")
print("=" * 60)
for s in ca_samples[:2]:
    print(f"\n[{s['dataset']}] {s['serial_number']} | iter {s['iteration']} | role: {s['role']}")
    print(f"  Context:              {s['context'][:120]}...")
    print(f"  Ground Truth:         {s['ground_truth']}")
    print(f"  Original Question(s): {s['previous_questions']}")
    print(f"  Corrected Question(s):{s['corrected_questions']}")
    print(f"  YOUR LABEL:           [ ] correct_fix  [ ] still_leaks  [ ] bad_question")

SAMPLE: False-Negative Check (checker said NO leak)
Task: Does the question actually leak the ground truth?

[PHEE] dev-435 | iter 0 | role: treatment-drug
  Context:      Pharmacokinetic modeling suggested that the patient's initial over-treatment was as reported and that the predicted maxi...
  Ground Truth: ['chloroquine']
  Question(s):  ['What is the name of the drug that was administered to the patient?']
  YOUR LABEL:   [ ] no_leak  [ ] actually_leaked

[PHEE] dev-69 | iter 0 | role: treatment
  Context:      CASE SUMMARY: A 58-year-old white woman developed fulminant liver failure while being treated with the macrolide antibio...
  Ground Truth: ['macrolide antibiotic clarithromycin for pneumonia']
  Question(s):  ['What specific medication was administered to the patient?', 'To which drug class does the administered medication belong?', 'What condition or disease was the medication prescribed to treat?']
  YOUR LABEL:   [ ] no_leak  [ ] actually_leaked


SAMPLE: Correction-Acc

## Dataset of auto interpretibility analsysis

In [ ]:
import json
import os

QG_DIR = os.path.join("..", "..", "Outputs", "qg")
OUTPUT_DIR = os.path.join("..", "..", "Outputs", "analysis")
DATASETS = ["CaseReportBench", "DiscourseEE", "MACCROBAT", "PHEE"]
MODEL_TAG = "gpt-oss-120b-zs-v0"
SPLIT = "dev"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_json(path):
    with open(path) as f:
        return json.load(f)


def build_lookup(records, key_fields=("serial-number", "role")):
    """Index records by (serial-number, role) for O(1) matching."""
    return {tuple(rec[k] for k in key_fields): rec for rec in records}


def consolidate_dataset(dataset):
    """
    Load the three dev-split QG files and merge on (serial-number, role):
      - schema-qg-*           → vanilaQ   (generic role-level)
      - loqa-qg-*             → zs-LoQ    (context-aware, zero-shot)
      - optimized_loqa-qg-*   → opt-LoQ   (optimized loqa)
    """
    qg_path = os.path.join(QG_DIR, dataset)

    schema_file = os.path.join(qg_path, f"schema-qg-{dataset}-{SPLIT}-{MODEL_TAG}.json")
    loqa_file   = os.path.join(qg_path, f"loqa-qg-{dataset}-{SPLIT}-{MODEL_TAG}.json")
    opt_file    = os.path.join(qg_path, f"optimized_loqa-qg-{dataset}-{SPLIT}-{MODEL_TAG}.json")

    for fp in [schema_file, loqa_file, opt_file]:
        if not os.path.exists(fp):
            print(f"  [SKIP] Missing: {fp}")
            return None

    schema_data = load_json(schema_file)
    loqa_data   = load_json(loqa_file)
    opt_data    = load_json(opt_file)

    print(f"  vanilaQ: {len(schema_data)} | zs-LoQ: {len(loqa_data)} | opt-LoQ: {len(opt_data)}")

    loqa_lookup = build_lookup(loqa_data)
    opt_lookup  = build_lookup(opt_data)

    consolidated = []
    matched = 0
    for rec in schema_data:
        key = (rec["serial-number"], rec["role"])

        entry = {k: v for k, v in rec.items() if k != "schema_questions"}
        schema_q = rec.get("schema_questions", "")
        entry["vanilaQ"] = [schema_q] if isinstance(schema_q, str) else schema_q

        loqa_rec = loqa_lookup.get(key)
        if loqa_rec:
            lq = loqa_rec.get("loqa_questions", [])
            entry["zs-LoQ"] = lq if isinstance(lq, list) else [lq]
        else:
            entry["zs-LoQ"] = []

        opt_rec = opt_lookup.get(key)
        if opt_rec:
            oq = opt_rec.get("optimized_loqa_questions", [])
            entry["opt-LoQ"] = oq if isinstance(oq, list) else [oq]
        else:
            entry["opt-LoQ"] = []

        if loqa_rec and opt_rec:
            matched += 1
        consolidated.append(entry)

    print(f"  Total: {len(consolidated)} | Fully matched: {matched}")
    return consolidated


# --- Run consolidation and save single combined file ---
all_records = []

for dataset in DATASETS:
    print(f"\n{'='*60}")
    print(f"Processing: {dataset}")
    print(f"{'='*60}")

    result = consolidate_dataset(dataset)
    if result is None:
        continue

    for rec in result:
        rec["dataset"] = dataset
    all_records.extend(result)

    sample = result[0]
    print(f"\n  Sample (first record):")
    print(f"    dataset:       {sample['dataset']}")
    print(f"    serial-number: {sample['serial-number']}")
    print(f"    role:          {sample['role']}")
    print(f"    vanilaQ:       {sample['vanilaQ']}")
    print(f"    zs-LoQ:        {sample['zs-LoQ']}")
    print(f"    opt-LoQ:       {sample['opt-LoQ']}")

# Save per-dataset files
for dataset in DATASETS:
    ds_records = [r for r in all_records if r["dataset"] == dataset]
    ds_path = os.path.join(OUTPUT_DIR, f"{dataset}-dev-questions-consolidated.json")
    with open(ds_path, "w") as f:
        json.dump(ds_records, f, indent=2, ensure_ascii=False)
    print(f"  {dataset}: {len(ds_records)} records → {os.path.basename(ds_path)}")

# Save combined file
out_path = os.path.join(OUTPUT_DIR, "all-dev-questions-consolidated.json")
with open(out_path, "w") as f:
    json.dump(all_records, f, indent=2, ensure_ascii=False)

print(f"\n{'='*60}")
print(f"Combined file: {os.path.abspath(out_path)}")
print(f"Total records: {len(all_records)}")
print(f"{'='*60}")